# Module 7 - Activity 3: SHAP + Explainer Dashboard (Titanic)

**Brief:** interact with the Titanic explainer dashboard - feature importances, model performance,
individual predictions, feature dependence - then discuss which features drive survival.

**Why this notebook:** `titanicexplainer.herokuapp.com` is dead (Heroku killed free dynos in 2022).
We rebuild the dashboard's four panels with SHAP directly - which is better anyway, because the
dashboard is just a UI over exactly these calls.

> **SHAP** (Lundberg 2017, on Shapley 1953): explain a prediction by computing **each feature's
> contribution** to it. The contributions are additive: `base_value + sum(shap) = prediction`.
> That additivity is what separates SHAP from every hand-wavy importance score.

| Dashboard panel | Scope | SHAP call |
|---|---|---|
| Feature importances | **global** | `shap.plots.bar` |
| Feature dependence | **global** | `shap.plots.scatter` |
| Individual prediction | **local** | `shap.plots.waterfall` |
| (all of it at once) | both | `shap.plots.beeswarm` |


In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
import shap

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix

SEED = 42
np.random.seed(SEED)
OUT = Path('outputs'); OUT.mkdir(exist_ok=True)
DATA = Path('dataset'); DATA.mkdir(exist_ok=True)

cache = DATA / 'titanic.csv'
if cache.exists():
    raw = pd.read_csv(cache)
else:
    raw = fetch_openml('titanic', version=1, as_frame=True).frame
    raw.to_csv(cache, index=False)

raw['survived'] = raw['survived'].astype(int)
print(f'{len(raw)} passengers | survival rate {raw.survived.mean():.1%}')
raw.head(3)

1309 passengers | survival rate 38.2%


/opt/homebrew/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
0,1,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0,0,24160,211.3375,B5,S,2,NaN,"St Louis, MO"
1,1,1,"Allison, Master. Hudson Trevor",male,0.9167,1,2,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,1,0,"Allison, Miss. Helen Loraine",female,2.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"


## Before any explaining: a leakage trap

This dataset ships with a column called `boat` - the lifeboat number. If you were **in a lifeboat,
you survived**. It is the outcome wearing a moustache.

Leave it in and you get a near-perfect model that has learnt nothing. Worse for this module: SHAP
will happily produce a **beautiful, confident, completely worthless explanation** of it. Let's prove
it, because this is the single most useful thing in the notebook.

In [2]:
def prep(df, cols):
    """Encode to numeric for the tree model.

    NB: test `is_numeric_dtype`, do NOT select on 'object'. fetch_openml returns
    sex/embarked as *category* dtype, and a category column silently survives an
    'object'-only filter -> pd.to_numeric coerces 'male' to NaN -> fillna(-1) makes
    the column CONSTANT -> the model never sees sex at all. (Ask me how I know.)
    """
    d = df[cols].copy()
    for c in d.columns:
        if not pd.api.types.is_numeric_dtype(d[c]):
            d[c] = d[c].astype('category').cat.codes   # strings/categories -> int, NaN -> -1
    return d.astype(float).fillna(-1)                  # numeric NaN (age) -> -1 sentinel

y = raw['survived']
leaky_cols = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'boat']
clean_cols = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']

for label, cols in [('WITH boat (leaky)', leaky_cols), ('WITHOUT boat (honest)', clean_cols)]:
    Xc = prep(raw, cols)
    xtr, xte, ytr, yte = train_test_split(Xc, y, test_size=.25,
                                          random_state=SEED, stratify=y)
    m = RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1).fit(xtr, ytr)
    auc = roc_auc_score(yte, m.predict_proba(xte)[:, 1])
    acc = accuracy_score(yte, m.predict(xte))
    print(f'{label:24s} accuracy={acc:.3f}  roc_auc={auc:.3f}')

print('\nThe leaky model looks brilliant and is useless: at prediction time (the ship is sinking)')
print('nobody knows their lifeboat number yet. Drop it - along with body/name/ticket/cabin.')

WITH boat (leaky)        accuracy=0.963  roc_auc=0.995


WITHOUT boat (honest)    accuracy=0.774  roc_auc=0.848

The leaky model looks brilliant and is useless: at prediction time (the ship is sinking)
nobody knows their lifeboat number yet. Drop it - along with body/name/ticket/cabin.


In [3]:
X = prep(raw, clean_cols)

# Guard rail: a constant column is a silently broken feature. Fail loudly instead.
dead = list(X.columns[X.nunique() <= 1])
assert not dead, f'constant (= broken) column(s): {dead}'
print('encoded feature check - unique values per column:')
print(X.nunique().to_string())

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=.25,
                                          random_state=SEED, stratify=y)

model = RandomForestClassifier(n_estimators=400, max_depth=6,
                               random_state=SEED, n_jobs=-1).fit(X_tr, y_tr)

pred = model.predict(X_te)
print('=== Panel: model performance ===')
print(f'accuracy {accuracy_score(y_te, pred):.3f} | '
      f'roc_auc {roc_auc_score(y_te, model.predict_proba(X_te)[:,1]):.3f}\n')
tn, fp, fn, tp = confusion_matrix(y_te, pred).ravel()
print(pd.DataFrame([[tn, fp], [fn, tp]],
                   index=['actual: died', 'actual: survived'],
                   columns=['pred: died', 'pred: survived']).to_string())

# sex was label-encoded: remember the mapping so we can read the plots
sex_codes = dict(enumerate(raw['sex'].astype('category').cat.categories))
print(f'\n(encoding: sex -> {sex_codes})')

encoded feature check - unique values per column:
pclass        3
sex           2
age          99
sibsp         7
parch         8
fare        282
embarked      4


=== Panel: model performance ===
accuracy 0.851 | roc_auc 0.892

                  pred: died  pred: survived
actual: died             185              18
actual: survived          31              94

(encoding: sex -> {0: 'female', 1: 'male'})


In [4]:
explainer = shap.TreeExplainer(model)
sv = explainer(X_te)
sv = sv[..., 1] if sv.values.ndim == 3 else sv   # keep the P(survived) output
print(f'SHAP values: {sv.values.shape}  (rows x features)')
print(f'base value (average prediction): {sv.base_values[0]:.3f}')

# The additivity guarantee - check it yourself, do not take it on faith
i = 0
recon = sv.base_values[i] + sv.values[i].sum()
actual = model.predict_proba(X_te.iloc[[i]])[0, 1]
print(f'\nbase + sum(shap) = {recon:.4f}')
print(f'model prediction = {actual:.4f}   <- they match. That is the Shapley guarantee.')

SHAP values: (328, 7)  (rows x features)
base value (average prediction): 0.382

base + sum(shap) = 0.0848
model prediction = 0.0848   <- they match. That is the Shapley guarantee.


## Panel 1 - Feature importances (**GLOBAL**)

Mean absolute SHAP per feature: *across all passengers*, how much did this feature move predictions?

> ⚠️ **This is the global side of the exam distinction.** It tells you nothing about any individual.

In [5]:
plt.figure()
shap.plots.bar(sv, max_display=8, show=False)
plt.title('Panel 1 - GLOBAL feature importance (mean |SHAP|)', fontsize=11)
plt.savefig(OUT / 'a3_global_importance.png', dpi=130, bbox_inches='tight')
plt.show()

imp = pd.Series(np.abs(sv.values).mean(0), index=X.columns).sort_values(ascending=False)
print(imp.round(4).to_string())

sex         0.1825
pclass      0.0650
fare        0.0441
embarked    0.0319
parch       0.0312
age         0.0197
sibsp       0.0155


/var/folders/c4/t_n58l316yl308hxjf5jlwsh0000gn/T/ipykernel_18238/3701699437.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Panel 2 - Beeswarm: importance **and direction**

The bar chart says *sex matters most*. It does not say **which sex**. The beeswarm does: every dot
is a passenger, colour is the feature's value, x-position is its SHAP contribution.

In [6]:
plt.figure()
shap.plots.beeswarm(sv, max_display=8, show=False)
plt.title('Panel 2 - beeswarm: each dot is a passenger', fontsize=11)
plt.savefig(OUT / 'a3_beeswarm.png', dpi=130, bbox_inches='tight')
plt.show()
print(f'sex encoding: {sex_codes}  ->  low (blue) = female, high (red) = male')
print('Read it: blue sex dots sit far RIGHT (positive SHAP = pushed toward survival).\n')

s = sv.values[:, list(X.columns).index('sex')]
print(f'mean SHAP(sex) | female: {s[X_te["sex"] == 0].mean():+.3f}   '
      f'male: {s[X_te["sex"] == 1].mean():+.3f}')
print('The bar chart could only tell you sex MATTERED. The beeswarm tells you WHICH WAY.')

sex encoding: {0: 'female', 1: 'male'}  ->  low (blue) = female, high (red) = male
Read it: blue sex dots sit far RIGHT (positive SHAP = pushed toward survival).

mean SHAP(sex) | female: +0.256   male: -0.140
The bar chart could only tell you sex MATTERED. The beeswarm tells you WHICH WAY.


/var/folders/c4/t_n58l316yl308hxjf5jlwsh0000gn/T/ipykernel_18238/3587898069.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Panel 3 - Individual predictions (**LOCAL**) ⭐

> **This is the local side of the exam distinction - and the whole reason SHAP exists.**
> Same model, same SHAP machinery, but now: *why did the model say THIS about THIS passenger?*

The waterfall starts at the base value (the average passenger) and each feature pushes the
prediction up or down until it lands on this passenger's score.

In [7]:
proba = model.predict_proba(X_te)[:, 1]
hi = int(np.argmax(proba))   # most confident survivor
lo = int(np.argmin(proba))   # most confident death

for tag, idx in [('MOST LIKELY TO SURVIVE', hi), ('LEAST LIKELY TO SURVIVE', lo)]:
    row = raw.loc[X_te.index[idx]]
    print(f'--- {tag} ---')
    print(f"  {row['name']}")
    print(f"  {row['sex']}, age {row['age']}, class {row['pclass']}, fare {row['fare']:.1f}")
    print(f"  model P(survived) = {proba[idx]:.3f}  |  actually survived: "
          f"{'YES' if row['survived'] else 'NO'}\n")

--- MOST LIKELY TO SURVIVE ---
  Meyer, Mrs. Edgar Joseph (Leila Saks)
  female, age nan, class 1, fare 82.2
  model P(survived) = 0.967  |  actually survived: YES

--- LEAST LIKELY TO SURVIVE ---
  Sage, Master. William Henry
  male, age 14.5, class 3, fare 69.5
  model P(survived) = 0.068  |  actually survived: NO



In [8]:
for tag, idx, fname in [('survivor', hi, 'a3_local_survivor.png'),
                        ('victim', lo, 'a3_local_victim.png')]:
    plt.figure()
    shap.plots.waterfall(sv[idx], max_display=8, show=False)
    name = raw.loc[X_te.index[idx], 'name'][:42]
    plt.title(f'Panel 3 - LOCAL: why the model predicted this for {name}', fontsize=10)
    plt.savefig(OUT / fname, dpi=130, bbox_inches='tight')
    plt.show()

print('Same model. Same SHAP. Two completely different explanations.')
print('THAT is what "local" means - and why a global importance bar could never tell you this.')

/var/folders/c4/t_n58l316yl308hxjf5jlwsh0000gn/T/ipykernel_18238/17287412.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Same model. Same SHAP. Two completely different explanations.
THAT is what "local" means - and why a global importance bar could never tell you this.


/var/folders/c4/t_n58l316yl308hxjf5jlwsh0000gn/T/ipykernel_18238/17287412.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Panel 4 - Feature dependence: how one feature's effect *varies*

A single number for 'age importance' hides that age helps the young and hurts the old - and that
its effect **depends on sex**. Dependence plots expose that interaction.

In [9]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
shap.plots.scatter(sv[:, 'age'], color=sv[:, 'sex'], ax=axes[0], show=False)
axes[0].set_title('age (coloured by sex)', fontsize=10)
shap.plots.scatter(sv[:, 'pclass'], color=sv[:, 'fare'], ax=axes[1], show=False)
axes[1].set_title('pclass (coloured by fare)', fontsize=10)
fig.suptitle('Panel 4 - feature dependence: effects are not constant', fontsize=11)
fig.tight_layout()
fig.savefig(OUT / 'a3_dependence.png', dpi=130, bbox_inches='tight')
plt.show()

# Quantify the age effect rather than eyeballing the scatter (-1 = imputed missing age)
age = X_te['age'].values
a_shap = sv.values[:, list(X.columns).index('age')]
real = age >= 0
band = pd.cut(age[real], [0, 12, 18, 35, 60, 100])
print('mean SHAP(age) by age band - real ages only:')
print(pd.Series(a_shap[real]).groupby(band, observed=True)
        .agg(['mean', 'count']).round(4).to_string())
print('\nDirection is right: children get a positive push, it decays monotonically with age.')
print('But look at the MAGNITUDE (~0.05) next to sex (~0.26). See the discussion below.')

mean SHAP(age) by age band - real ages only:
             mean  count
(0, 12]    0.0514     26
(12, 18]   0.0171     18
(18, 35]   0.0045    140
(35, 60]  -0.0291     74
(60, 100] -0.0616      6

Direction is right: children get a positive push, it decays monotonically with age.
But look at the MAGNITUDE (~0.05) next to sex (~0.26). See the discussion below.


/var/folders/c4/t_n58l316yl308hxjf5jlwsh0000gn/T/ipykernel_18238/435721152.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Discussion (the brief's questions)

**"Which parameters matter most for survival?"**
`sex` dominates and it is not close - mean |SHAP| ~0.18 against ~0.07 for the next feature. Being
female adds **+0.26** to the predicted survival probability; being male subtracts **0.14**. Then come
`pclass` and `fare` - the same thing wearing two hats: **wealth**.

The interesting part is what the numbers *refuse* to say. The folk summary is *"women and children
first"*, but `age` ranks **6th of 7** (mean |SHAP| 0.020). Its *direction* is right - binning the
SHAP values by age band gives children (0-12) **+0.051** and the elderly (60+) **-0.062**, a clean
monotone decay - but its *magnitude* is small. What the model actually learnt from the data is
**"women first, money second, children a distant third"**.

That gap between the story you expected and the numbers you got is the entire value of SHAP. A
narrative would have sailed straight past it.

**"What is the relationship between features and output?"**
Not linear and not independent - Panel 4 shows age's effect **depends on** sex. This is exactly why
a single importance number is not an explanation.

**"Should consumers have a dashboard whenever an ML decision affects them?"**
The honest answer from this notebook is *yes, but a dashboard is not enough*:

- The **leakage demo** is the argument. A model trained on `boat` scores ~0.97 AUC and SHAP explains
  it *fluently*. The explanation would be confident, well-rendered, and worthless. **A dashboard
  cannot tell you the model is asking the wrong question** - that is Aha's "a wrong explanation is
  worse than none" (Res. 4), demonstrated rather than asserted.
- SHAP explains **what the model did**, never **whether it should have**. Here the top feature is
  `sex`. On the Titanic that is history. In a loan model it is a lawsuit.

So: ship the dashboard, but ship it *with* the governance around it (Sowden, Res. 5). The
explanation is the deliverable; the accountability for it is the point.

---

### The one table to remember

| | **Global** (the model) | **Local** (one prediction) |
|---|---|---|
| **Question** | what does it do in general? | why did it say *this*, about *me*? |
| **Method** | `shap.plots.bar`, beeswarm, permutation importance, PDP | `shap.plots.waterfall`, LIME, counterfactual/CEM |
| **Panel** | 1, 2, 4 | 3 |
| **Audience** | the modeller, the auditor | the person the decision happened to |

**Carry into Assessment 2:** Panels 1-3 on your wine classifier turn the evaluation section from
*"the model got 0.89"* into *"the model got 0.89, and here is what it learnt, and here is why it
misjudged this particular wine."* That is the difference between a pass and a distinction.